In [ ]:
# set_sequence_classifier.py

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

# ==============================================================================
# 1. Configuration Dictionary
# ==============================================================================
# Centralized configuration for all hyperparameters and settings.
config = {
    # --- Data Configuration ---
    "data": {
        "test_size": 0.2,
        "sequence_length": 50,
        "target_col": "target",
        "unit_id_col": "unit_id",
        "time_col": "time_period"
    },
    # --- Model Architecture ---
    "model": {
        "d_model": 128,
        "d_set_summary": 16,
        "phi_hidden_dim": 256,
        "rho_hidden_dim": 64,
        "psi_hidden_dim": 128,
        "n_set_seq_layers": 4,
        "n_seq_layers": 2,
        "dropout": 0.1,
        "long_conv_kernel_size": 32,
    },
    # --- Training Configuration ---
    "training": {
        "loss_function": "gmean", # Options: "bce", "gmean"
        "learning_rate": 0.001,
        "batch_size": 32, # Reduced batch size to accommodate larger tensors
        "n_epochs": 20,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "scaler": "standard",
    },
    # --- Cross-Validation ---
    "cross_val": {
        "n_splits": 5
    }
}

# ==============================================================================
# 2. Model Components & Architecture
# ==============================================================================

class GMeanLoss(nn.Module):
    """
    A differentiable loss function to maximize the G-mean.
    Loss = 1 - G-mean = 1 - sqrt(Sensitivity * Specificity).
    """
    def __init__(self, epsilon=1e-8):
        super().__init__()
        self.epsilon = epsilon

    def forward(self, logits, labels):
        preds = torch.sigmoid(logits)
        labels = labels.float()

        tp = torch.sum(preds * labels)
        sensitivity = tp / (torch.sum(labels) + self.epsilon)
        specificity = torch.sum((1 - preds) * (1 - labels)) / (torch.sum(1 - labels) + self.epsilon)

        g_mean = torch.sqrt(sensitivity * specificity + self.epsilon)
        return 1 - g_mean

class LongConv(nn.Module):
    """A simple 1D causal convolution layer."""
    def __init__(self, d_model, kernel_size, dropout):
        super().__init__()
        self.conv = nn.Conv1d(
            in_channels=d_model,
            out_channels=d_model,
            kernel_size=kernel_size,
            padding=kernel_size - 1,
            groups=d_model
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.conv(x)
        x = x[:, :, :-(self.conv.kernel_size[0] - 1)]
        x = x.transpose(1, 2)
        return self.dropout(x)


class SetSequenceLayer(nn.Module):
    """Implements a single Set-Sequence layer."""
    def __init__(self, config):
        super().__init__()
        d_model = config["model"]["d_model"]
        d_set_summary = config["model"]["d_set_summary"]
        phi_hidden = config["model"]["phi_hidden_dim"]
        rho_hidden = config["model"]["rho_hidden_dim"]
        psi_hidden = config["model"]["psi_hidden_dim"]
        kernel_size = config["model"]["long_conv_kernel_size"]
        dropout = config["model"]["dropout"]

        self.phi = nn.Sequential(nn.Linear(d_model, phi_hidden), nn.ReLU(), nn.Linear(phi_hidden, d_model))
        self.rho = nn.Sequential(nn.Linear(d_model, rho_hidden), nn.ReLU(), nn.Linear(rho_hidden, d_set_summary))
        self.psi = nn.Sequential(nn.Linear(d_model + d_set_summary, psi_hidden), nn.ReLU(), nn.Linear(psi_hidden, d_model))
        self.seq_layer = LongConv(d_model, kernel_size, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        batch_size, num_units, seq_len, d_model = x.shape

        x_reshaped = x.view(batch_size * num_units, seq_len, d_model)
        phi_x = self.phi(x_reshaped).view(batch_size, num_units, seq_len, d_model)
        mean_phi_x = torch.mean(phi_x, dim=1)
        set_summary = self.rho(mean_phi_x)

        set_summary_expanded = set_summary.unsqueeze(1).expand(-1, num_units, -1, -1)
        augmented_x = torch.cat([x, set_summary_expanded], dim=-1)
        augmented_x_reshaped = augmented_x.view(batch_size * num_units, seq_len, -1)
        psi_out = self.psi(augmented_x_reshaped)

        res_x = x.view(batch_size * num_units, seq_len, d_model)
        processed_x = self.norm1(res_x + psi_out)
        seq_out = self.seq_layer(processed_x)
        final_out = self.norm2(processed_x + seq_out)

        return final_out.view(batch_size, num_units, seq_len, d_model)


class SetSequenceClassifier(nn.Module):
    """The full Set-Sequence model for per-unit classification."""
    def __init__(self, config, n_features):
        super().__init__()
        d_model = config["model"]["d_model"]
        n_set_seq_layers = config["model"]["n_set_seq_layers"]
        n_seq_layers = config["model"]["n_seq_layers"]
        kernel_size = config["model"]["long_conv_kernel_size"]
        dropout = config["model"]["dropout"]

        self.input_projection = nn.Linear(n_features, d_model)
        self.set_seq_layers = nn.ModuleList([SetSequenceLayer(config) for _ in range(n_set_seq_layers)])
        self.final_seq_layers = nn.ModuleList([LongConv(d_model, kernel_size, dropout) for _ in range(n_seq_layers)])
        self.classifier_head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))

    def forward(self, x):
        """
        Args:
            x (torch.Tensor): Input of shape (batch_size, num_units, seq_len, n_features)
        Returns:
            torch.Tensor: Logits of shape (batch_size, num_units)
        """
        batch_size, num_units, seq_len, _ = x.shape

        # 1. Project input features to d_model
        x = self.input_projection(x)

        # 2. Pass through Set-Sequence layers
        for layer in self.set_seq_layers:
            x = layer(x)

        # 3. Reshape to process all units' sequences in one go
        # (batch * units, seq_len, d_model)
        x = x.view(batch_size * num_units, seq_len, -1)

        # 4. Pass through final sequence-only layers
        for layer in self.final_seq_layers:
            x = x + layer(x)

        # 5. Use the last time step's output for classification
        last_time_step = x[:, -1, :]

        # 6. Get logits from the classifier head
        logits = self.classifier_head(last_time_step)

        # 7. Reshape back to (batch_size, num_units)
        return logits.view(batch_size, num_units)

# ==============================================================================
# 3. Data Preparation and Utilities
# ==============================================================================

def create_sequences(df, config):
    """
    Transforms the DataFrame into sequences and per-unit targets.
    X shape: (num_sequences, num_units, seq_len, n_features)
    y shape: (num_sequences, num_units)
    """
    seq_len = config["data"]["sequence_length"]
    unit_col = config["data"]["unit_id_col"]
    time_col = config["data"]["time_col"]
    target_col = config["data"]["target_col"]

    feature_cols = [c for c in df.columns if c not in [unit_col, time_col, target_col]]
    n_features = len(feature_cols)

    df = df.sort_values(by=[time_col, unit_col])

    sequences, targets = [], []
    unique_times = df[time_col].unique()

    for t in range(seq_len, len(unique_times)):
        start_time, end_time, target_time = unique_times[t - seq_len], unique_times[t - 1], unique_times[t]

        sequence_df = df[(df[time_col] >= start_time) & (df[time_col] <= end_time)]
        target_df = df[df[time_col] == target_time]

        # Get the units present in this time window
        units_in_window = sequence_df[unit_col].unique()

        # Pivot features
        seq_pivot = sequence_df.pivot(index=unit_col, columns=time_col, values=feature_cols).fillna(0)

        # Pivot targets
        target_pivot = target_df.pivot(index=unit_col, columns=time_col, values=target_col)

        # Align units between features and targets, fill missing targets with 0
        seq_pivot, target_pivot = seq_pivot.align(target_pivot, join='left', axis=0, fill_value=0)

        num_units = len(seq_pivot)
        if num_units == 0: continue

        try:
            seq_array = seq_pivot.values.reshape(num_units, seq_len, n_features)
            target_array = target_pivot.values.flatten()

            sequences.append(seq_array)
            targets.append(target_array)
        except ValueError:
            continue

    # Note: Sequences can have different numbers of units. This requires custom padding/batching.
    # For simplicity here, we filter for sequences with the most common number of units.
    if not sequences: return np.array([]), np.array([])

    unit_counts = [s.shape[0] for s in sequences]
    if not unit_counts: return np.array([]), np.array([])

    most_common_n_units = max(set(unit_counts), key=unit_counts.count)

    X_filtered = [sequences[i] for i, count in enumerate(unit_counts) if count == most_common_n_units]
    y_filtered = [targets[i] for i, count in enumerate(unit_counts) if count == most_common_n_units]

    if not X_filtered: return np.array([]), np.array([])

    return np.stack(X_filtered), np.stack(y_filtered)


def get_scaler(name):
    if name == "standard": return StandardScaler()
    if name == "minmax": return MinMaxScaler()
    return None

def get_criterion(name):
    if name == "bce": return nn.BCEWithLogitsLoss()
    if name == "gmean": return GMeanLoss()
    raise ValueError(f"Unknown loss function: {name}")

def calculate_gmean(labels, preds, epsilon=1e-8):
    if len(np.unique(labels)) < 2: return 0.0
    cm = confusion_matrix(labels, np.round(preds))
    if cm.shape != (2, 2): return 0.0
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn + epsilon)
    specificity = tn / (tn + fp + epsilon)
    return np.sqrt(sensitivity * specificity)

# ==============================================================================
# 4. Training and Evaluation Loop
# ==============================================================================

def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for X_batch, y_batch in dataloader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch) # Shape: (batch, units)

        # Flatten outputs and targets for loss calculation
        loss = criterion(outputs.view(-1), y_batch.view(-1).float())

        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)

            # Flatten for loss and metrics
            flat_outputs = outputs.view(-1)
            flat_labels = y_batch.view(-1).float()

            loss = criterion(flat_outputs, flat_labels)
            total_loss += loss.item()

            all_preds.extend(torch.sigmoid(flat_outputs).cpu().numpy())
            all_labels.extend(flat_labels.cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    auc = roc_auc_score(all_labels, all_preds)
    acc = accuracy_score(all_labels, np.round(all_preds))
    gmean = calculate_gmean(all_labels, all_preds)

    return avg_loss, auc, acc, gmean

# ==============================================================================
# 5. Main Execution
# ==============================================================================

def main():
    print("--- Set-Sequence Model for Per-Unit Classification ---")
    print(f"Using device: {config['training']['device']}")
    print(f"Optimizing with loss function: {config['training']['loss_function'].upper()}")

    print("Generating sample data...")
    n_units, n_time_periods, n_features = 50, 500, 10
    data = []
    for unit in range(n_units):
        for time in range(n_time_periods):
            row = {'unit_id': unit, 'time_period': time}
            features = np.sin(time / 50 + unit) + np.random.randn(n_features) * 0.5
            for i, f_val in enumerate(features): row[f'feature_{i}'] = f_val
            row['target'] = 1 if (features[0] + np.sin(time/20)) > 0.8 else 0
            data.append(row)
    df = pd.DataFrame(data)
    print(f"Sample data created. Target distribution:\n{df['target'].value_counts(normalize=True)}")

    feature_cols = [c for c in df.columns if isinstance(c, str) and c.startswith('feature')]
    time_col = config['data']['time_col']
    test_split_time = df[time_col].unique()[int(len(df[time_col].unique()) * (1 - config['data']['test_size']))]
    df_train_val, df_test = df[df[time_col] < test_split_time], df[df[time_col] >= test_split_time]

    scaler = get_scaler(config['training']['scaler'])
    if scaler:
        print(f"Applying {config['training']['scaler']} scaling...")
        df_train_val.loc[:, feature_cols] = scaler.fit_transform(df_train_val[feature_cols])
        df_test.loc[:, feature_cols] = scaler.transform(df_test[feature_cols])

    print("\nStarting walk-forward cross-validation...")
    tscv = TimeSeriesSplit(n_splits=config['cross_val']['n_splits'])
    fold_results = []
    time_periods = df_train_val[time_col].unique()

    for fold, (train_indices, val_indices) in enumerate(tscv.split(time_periods)):
        print(f"\n--- Fold {fold + 1}/{config['cross_val']['n_splits']} ---")
        df_train_fold = df_train_val[df_train_val[time_col].isin(time_periods[train_indices])]
        df_val_fold = df_train_val[df_train_val[time_col].isin(time_periods[val_indices])]

        X_train, y_train = create_sequences(df_train_fold, config)
        X_val, y_val = create_sequences(df_val_fold, config)

        if X_train.shape[0] == 0 or X_val.shape[0] == 0:
            print("Skipping fold due to insufficient data to create sequences.")
            continue

        train_loader = DataLoader(TensorDataset(torch.from_numpy(X_train).float(), torch.from_numpy(y_train).long()), batch_size=config['training']['batch_size'], shuffle=True)
        val_loader = DataLoader(TensorDataset(torch.from_numpy(X_val).float(), torch.from_numpy(y_val).long()), batch_size=config['training']['batch_size'])

        model = SetSequenceClassifier(config, n_features=len(feature_cols)).to(config['training']['device'])
        optimizer = optim.Adam(model.parameters(), lr=config['training']['learning_rate'])
        criterion = get_criterion(config['training']['loss_function'])

        for epoch in range(config['training']['n_epochs']):
            train_loss = train_epoch(model, train_loader, optimizer, criterion, config['training']['device'])
            val_loss, val_auc, val_acc, val_gmean = evaluate(model, val_loader, criterion, config['training']['device'])
            if (epoch + 1) % 5 == 0:
                 print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val AUC: {val_auc:.4f} | Val G-mean: {val_gmean:.4f}")

        fold_results.append({'auc': val_auc, 'acc': val_acc, 'gmean': val_gmean})

    print("\n--- Final Evaluation on Test Set ---")
    print("Retraining model on full train/val data...")
    X_train_full, y_train_full = create_sequences(df_train_val, config)
    X_test, y_test = create_sequences(df_test, config)

    if X_train_full.shape[0] == 0 or X_test.shape[0] == 0:
        print("Cannot perform final evaluation due to insufficient data.")
        return

    train_full_loader = DataLoader(TensorDataset(torch.from_numpy(X_train_full).float(), torch.from_numpy(y_train_full).long()), batch_size=config['training']['batch_size'], shuffle=True)
    test_loader = DataLoader(TensorDataset(torch.from_numpy(X_test).float(), torch.from_numpy(y_test).long()), batch_size=config['training']['batch_size'])

    final_model = SetSequenceClassifier(config, n_features=len(feature_cols)).to(config['training']['device'])
    optimizer = optim.Adam(final_model.parameters(), lr=config['training']['learning_rate'])
    criterion = get_criterion(config['training']['loss_function'])

    for epoch in range(config['training']['n_epochs']):
        train_loss = train_epoch(final_model, train_full_loader, optimizer, criterion, config['training']['device'])
        if (epoch + 1) % 5 == 0: print(f"Retraining Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f}")

    test_loss, test_auc, test_acc, test_gmean = evaluate(final_model, test_loader, criterion, config['training']['device'])

    print("\n--- Results Summary ---")
    if fold_results:
        avg_cv_gmean = np.mean([r['gmean'] for r in fold_results])
        print(f"Average Cross-Validation G-mean: {avg_cv_gmean:.4f}")

    print(f"\nFinal Test Set G-mean: {test_gmean:.4f}")
    print(f"Final Test Set AUC: {test_auc:.4f}")
    print(f"Final Test Set Accuracy: {test_acc:.4f}")


if __name__ == "__main__":
    main()
